In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import src.utils.feature_utils as fe

In [3]:
train_df = pd.read_csv("../data/raw/train.csv", index_col='id')
test_df = pd.read_csv("../data/raw/test.csv", index_col='id')
orig_df = pd.read_csv('../data/raw/loan_dataset_20000.csv')

In [4]:
TARGET = 'loan_paid_back'
y = train_df[TARGET]

# KNN "identity theft"

In [5]:
shared_cols = ['loan_amount', 'interest_rate', 'debt_to_income_ratio', 'credit_score', 'annual_income']
borrow_cols = ['age', 'current_balance', 'installment', 'total_credit_limit', 'loan_term', 'num_of_delinquencies']

In [6]:
train_df, test_df = fe.borrow_features_with_knn(train_df, test_df, orig_df, shared_cols, borrow_cols)

In [7]:
combined = pd.concat([
        train_df.drop(columns=[TARGET], errors='ignore'),
        test_df
    ])

NUMS = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
CATS = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']

# Domain knowledge

In [8]:
financial_risk_features = fe.make_financial_risk_features(combined)

demographic_features = fe.make_demographic_features(combined)

combined = pd.concat([combined,financial_risk_features, demographic_features], axis=1)

# Discretization of numerical features and rounding

In [9]:
features_to_discretize = ['annual_income', 'loan_amount']

qcut_features = fe.make_quantile_binned_features(combined, features_to_discretize, n_bins=10_000)

uniform_cut_features = fe.make_uniform_binned_features(combined, features_to_discretize, n_bins=10_000)

log_cut = fe.make_log_binned_features(combined, features_to_discretize, n_bins=10_000)

round_half_features = fe.make_rounded_halved_features(combined, features_to_discretize[0:2])

combined = pd.concat([combined, qcut_features, uniform_cut_features, log_cut, round_half_features], axis=1)

orig_round_half_df = fe.make_rounded_halved_features(orig_df, features_to_discretize)
orig_df = pd.concat([orig_df, orig_round_half_df], axis=1)

# Count encoding

In [10]:
high_cardinality_features = ['employment_status', 'loan_purpose', 'grade_subgrade']


count_df = fe.make_count_features(combined, high_cardinality_features)
combined = pd.concat([combined, count_df], axis=1)

# Digits

In [11]:
deep_digits_features = ['annual_income', 'debt_to_income_ratio', 'loan_amount', 'interest_rate']

digits_df = fe.make_deep_digits_features(combined, deep_digits_features)

combined = pd.concat([combined, digits_df], axis=1)

# Over/undersampling

In [12]:
ratio_features = ['loan_amount', 'annual_income', 'interest_rate', 'debt_to_income_ratio']

density_df = fe.make_density_ratio_features(combined, orig_df, ratio_features)

combined = pd.concat([combined, density_df], axis=1)

# Scorecard

In [13]:
score_card_feature = fe.make_custom_scorecard(combined)

combined = pd.concat([combined, score_card_feature], axis=1)

# Prepare for target encoding using original data

In [14]:
categorical_columns = high_cardinality_features + round_half_features.columns.tolist()

X_train_full = combined.iloc[:len(train_df)].copy()
X_test_full = combined.iloc[len(train_df):].copy()

orig_te_mapped_train = pd.DataFrame(index=X_train_full.index)
orig_te_mapped_test = pd.DataFrame(index=X_test_full.index)

pseudo_targets = ['debt_to_income_ratio', 'interest_rate', 'annual_income', 'loan_amount']

# Target encoding using original data with pseudo targets

In [15]:
for col in categorical_columns:
    # Global mean for the main target
    global_target_mean = orig_df[TARGET].mean()
    orig_mean = orig_df.groupby(col)[TARGET].mean()

    orig_te_mapped_train[f'TE_orig_{col}'] = X_train_full[col].map(orig_mean).fillna(global_target_mean).astype(
            'float32')
    orig_te_mapped_test[f'TE_orig_{col}'] = X_test_full[col].map(orig_mean).fillna(global_target_mean).astype(
            'float32')

    for p_target in pseudo_targets:
        # Global mean for the pseudo target
        global_pseudo_mean = orig_df[p_target].mean()
        pseudo_mean = orig_df.groupby(col)[p_target].mean()

        col_name = f'PseudoTE_{p_target}_by_{col}'
        orig_te_mapped_train[col_name] = X_train_full[col].map(pseudo_mean).fillna(global_pseudo_mean).astype(
                'float32')
        orig_te_mapped_test[col_name] = X_test_full[col].map(pseudo_mean).fillna(global_pseudo_mean).astype(
                'float32')

In [16]:
X_train_full = pd.concat([X_train_full, orig_te_mapped_train], axis=1)
X_test_full = pd.concat([X_test_full, orig_te_mapped_test], axis=1)

# Exporting processed data

In [17]:
X_processed = pd.concat([X_train_full, y], axis=1)
X_test_processed = X_test_full

In [18]:
X_processed.columns.tolist()

['annual_income',
 'debt_to_income_ratio',
 'credit_score',
 'loan_amount',
 'interest_rate',
 'gender',
 'marital_status',
 'education_level',
 'employment_status',
 'loan_purpose',
 'grade_subgrade',
 'knn_orig_age',
 'knn_orig_current_balance',
 'knn_orig_installment',
 'knn_orig_total_credit_limit',
 'knn_orig_loan_term',
 'knn_orig_num_of_delinquencies',
 'knn_orig_distance',
 'default_risk',
 'expected_loss',
 'expected_return',
 'risk_adjusted_return',
 'character_proxy',
 'annual_income_quantile_binned',
 'loan_amount_quantile_binned',
 'annual_income_uniform_binned',
 'loan_amount_uniform_binned',
 'annual_income_log_binned',
 'loan_amount_log_binned',
 'annual_income_round_half',
 'loan_amount_round_half',
 'CE_employment_status',
 'CE_loan_purpose',
 'CE_grade_subgrade',
 'annual_income_int_digit_0',
 'annual_income_int_digit_1',
 'annual_income_int_digit_2',
 'annual_income_int_digit_3',
 'annual_income_int_digit_4',
 'annual_income_int_digit_5',
 'annual_income_dec_digit_0

In [19]:
X_processed.head()

,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,...,PseudoTE_debt_to_income_ratio_by_annual_income_round_half,PseudoTE_interest_rate_by_annual_income_round_half,PseudoTE_annual_income_by_annual_income_round_half,PseudoTE_loan_amount_by_annual_income_round_half,TE_orig_loan_amount_round_half,PseudoTE_debt_to_income_ratio_by_loan_amount_round_half,PseudoTE_interest_rate_by_loan_amount_round_half,PseudoTE_annual_income_by_loan_amount_round_half,PseudoTE_loan_amount_by_loan_amount_round_half,loan_paid_back
id,,,,,,,,,,,,,,,,,,,,,
0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,...,0.400,11.660000,29367.990234,29114.279297,0.666667,0.196333,13.763333,33334.535156,2528.429932,1.0
1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,...,0.217,8.680000,22108.019531,12649.160156,1.000000,0.154500,10.010000,84079.382812,4593.089844,0.0
2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,...,0.359,11.970000,49566.050781,10127.290039,1.000000,0.155500,13.817500,65977.218750,17004.626953,1.0
3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,...,0.282,12.165000,46858.554688,20822.259766,1.000000,0.197500,12.160000,30182.714844,4682.677734,1.0
4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,...,0.186,11.506667,25496.085938,14190.513672,0.500000,0.178250,11.617500,33181.789062,12184.275391,1.0


In [20]:
cats = X_processed.select_dtypes(include=['object','string', 'category']).columns

importances_df = pd.read_csv('../logs/20260328_001426_catboost_exodia_importances.csv')

importance_threshold = 0.1
features_to_drop = importances_df[importances_df['importance'] < importance_threshold]['feature'].tolist()


X_processed = X_processed.drop(columns=features_to_drop, errors='ignore')
X_test_processed = X_test_processed.drop(columns=features_to_drop, errors='ignore')

for col in cats:
    X_processed[col] = X_processed[col].astype(str)
    X_test_processed[col] = X_test_processed[col].astype(str)

In [26]:
X_processed.columns.tolist()

['annual_income',
 'debt_to_income_ratio',
 'credit_score',
 'loan_amount',
 'interest_rate',
 'gender',
 'marital_status',
 'education_level',
 'employment_status',
 'loan_purpose',
 'grade_subgrade',
 'knn_orig_age',
 'knn_orig_current_balance',
 'knn_orig_installment',
 'knn_orig_total_credit_limit',
 'knn_orig_num_of_delinquencies',
 'knn_orig_distance',
 'default_risk',
 'expected_loss',
 'expected_return',
 'risk_adjusted_return',
 'character_proxy',
 'annual_income_quantile_binned',
 'loan_amount_quantile_binned',
 'annual_income_uniform_binned',
 'loan_amount_uniform_binned',
 'annual_income_log_binned',
 'loan_amount_log_binned',
 'annual_income_round_half',
 'loan_amount_round_half',
 'CE_employment_status',
 'CE_grade_subgrade',
 'annual_income_int_digit_1',
 'annual_income_int_digit_2',
 'annual_income_int_digit_3',
 'annual_income_int_digit_4',
 'annual_income_int_digit_5',
 'annual_income_dec_digit_0',
 'annual_income_dec_digit_1',
 'annual_income_dec_digit_2',
 'annual_i

In [24]:
X_processed.to_parquet(f'../data/processed/FE_train.parquet', index=False)
X_test_processed.to_parquet(f'../data/processed/FE_test.parquet')